In [0]:
%sql
CREATE OR REPLACE TABLE university_cata.gold.student_insights AS
SELECT
    s.id_student,
    s.code_module,
    s.code_presentation,

    -- student info
    s.gender,
    s.region,
    s.highest_education,
    s.age_band,
    s.imd_band,
    s.num_of_prev_attempts,
    s.studied_credits,
    s.disability,
    s.final_result,

    -- engagement (aggregated)
    a.total_clicks,

    -- performance
    p.avg_score,
    p.weighted_score,

    -- lifecycle
    r.date_registration,
    r.date_unregistration,

    -- course info
    m.module_presentation_length

FROM university_cata.silver.dim_students s

LEFT JOIN university_cata.silver.student_activity a
ON s.id_student = a.id_student
AND s.code_module = a.code_module
AND s.code_presentation = a.code_presentation

LEFT JOIN university_cata.silver.student_performance p
ON s.id_student = p.id_student
AND s.code_module = p.code_module
AND s.code_presentation = p.code_presentation

LEFT JOIN university_cata.silver.student_registration r
ON s.id_student = r.id_student
AND s.code_module = r.code_module
AND s.code_presentation = r.code_presentation

LEFT JOIN university_cata.silver.dim_module m
ON s.code_module = m.code_module
AND s.code_presentation = m.code_presentation;

The duplications checks 

In [0]:
%sql
SELECT
    id_student,
    code_module,
    code_presentation,
    COUNT(*) AS cnt
FROM university_cata.gold.student_insights
GROUP BY
    id_student,
    code_module,
    code_presentation
HAVING cnt > 1;

In [0]:
%sql
SELECT
    COUNT(*) AS total,
    COUNT(weighted_score) AS non_null_scores
FROM university_cata.gold.student_insights;

In [0]:
%sql
CREATE OR REPLACE TABLE university_cata.gold.student_insights AS
SELECT
    s.id_student,
    s.code_module,
    s.code_presentation,

    s.gender,
    s.region,
    s.highest_education,
    s.age_band,
    s.imd_band,
    s.num_of_prev_attempts,
    s.studied_credits,
    s.disability,
    s.final_result,

    -- SUM clicks here to avoid duplication
    COALESCE(SUM(a.total_clicks), 0) AS total_clicks,
    COALESCE(p.weighted_score, 0) AS weighted_score,

    p.avg_score,
    r.date_registration,
    r.date_unregistration,
    c.module_presentation_length

FROM university_cata.silver.dim_students s

LEFT JOIN university_cata.silver.student_activity_agg a
    ON s.id_student = a.id_student
    AND s.code_module = a.code_module
    AND s.code_presentation = a.code_presentation

LEFT JOIN university_cata.silver.student_performance p
    ON s.id_student = p.id_student
    AND s.code_module = p.code_module
    AND s.code_presentation = p.code_presentation

LEFT JOIN university_cata.silver.student_registration r
    ON s.id_student = r.id_student
    AND s.code_module = r.code_module
    AND s.code_presentation = r.code_presentation

LEFT JOIN university_cata.silver.dim_module c
    ON s.code_module = c.code_module
    AND s.code_presentation = c.code_presentation

GROUP BY
    s.id_student,
    s.code_module,
    s.code_presentation,
    s.gender,
    s.region,
    s.highest_education,
    s.age_band,
    s.imd_band,
    s.num_of_prev_attempts,
    s.studied_credits,
    s.disability,
    s.final_result,
    p.avg_score,
    p.weighted_score,
    r.date_registration,
    r.date_unregistration,
    c.module_presentation_length;

In [0]:
%sql
SELECT
    id_student,
    code_module,
    code_presentation,
    COUNT(*) AS cnt
FROM university_cata.gold.student_insights
GROUP BY
    id_student,
    code_module,
    code_presentation
HAVING cnt > 1;

In [0]:
%sql
SELECT
    *
FROM university_cata.gold.student_insights
where weighted_score is null;

In [0]:
%sql
SELECT COUNT(*) FROM university_cata.gold.student_insights;

In [0]:
%sql
SELECT COUNT(DISTINCT id_student, code_module, code_presentation) as value_check FROM university_cata.gold.student_insights;